# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Lacenedihia/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I use a shallow Decision Tree as the primary model because the lane is a yes/no refresh-risk label and the tree produces readable rules. I also fit Logistic Regression as a stable linear reference and Random Forest as a bounded stronger comparison. The target is `is_declining_label`, defined as `trend_direction == "down"`; `trend_direction`, `trend_pct`, IDs, and any label-derived fields are excluded from features. Models are judged by Precision@50 first because the workflow is a small ranked review queue, with average precision and ROC AUC as secondary measures.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = Path("../..")
DATA_PATH = ROOT / "data" / "raw" / "content_refresh_anonymized.csv"
RANDOM_STATE = 42

raw = pd.read_csv(DATA_PATH)
required = ["content_id", "client_id", "impressions_90d", "sessions_90d", "content_age_days", "trend_direction"]
missing = [column for column in required if column not in raw.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

numeric_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
categorical_features = [
    "competition_level", "content_type", "main_intent", "age_tier", "freshness_tier",
    "word_count_tier", "impression_tier", "position_tier",
]
for column in numeric_features + ["impressions_90d", "sessions_90d", "clicks_90d", "ai_sessions_90d"]:
    if column in raw:
        raw[column] = pd.to_numeric(raw[column], errors="coerce")
for column in categorical_features:
    if column not in raw:
        raw[column] = "unknown"
    raw[column] = raw[column].fillna("unknown").astype(str).replace({"": "unknown", "nan": "unknown"})

frame = raw.copy()
for column in ["impressions_90d", "sessions_90d", "clicks_90d", "ai_sessions_90d"]:
    frame[column] = frame[column].replace([np.inf, -np.inf], np.nan).fillna(0)
frame = frame[(frame["impressions_90d"] > 0) & (frame["content_age_days"] >= 90)].drop_duplicates("content_id").reset_index(drop=True)
frame["is_declining_label"] = frame["trend_direction"].astype(str).str.lower().eq("down").astype(int)
frame["log_impressions_90d"] = np.log1p(frame["impressions_90d"])
frame["log_clicks_90d"] = np.log1p(frame["clicks_90d"])
frame["log_sessions_90d"] = np.log1p(frame["sessions_90d"])
frame["log_ai_sessions_90d"] = np.log1p(frame["ai_sessions_90d"])

numeric_frame = frame[numeric_features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
categorical_frame = frame[categorical_features].fillna("unknown").astype(str)
X = pd.concat([numeric_frame.reset_index(drop=True), pd.get_dummies(categorical_frame, prefix=categorical_features, dtype=float).reset_index(drop=True)], axis=1)
y = frame["is_declining_label"].astype(int)

def precision_at_k(y_true, scores, k):
    ranked = pd.DataFrame({"y": np.asarray(y_true), "score": np.asarray(scores)}).sort_values("score", ascending=False).head(k)
    return float(ranked["y"].mean()) if len(ranked) else 0.0

def normalize(values):
    values = pd.to_numeric(values, errors="coerce").fillna(0)
    span = values.max() - values.min()
    return (values - values.min()) / span if span else values * 0

visibility = frame["log_impressions_90d"].rank(pct=True).fillna(0)
freshness = frame["days_since_last_update"].rank(pct=True).fillna(0)
position = (1 - normalize(frame["avg_position"].clip(lower=1, upper=50))) * visibility * frame["avg_position"].gt(0)
depth_gap = (1 - frame["word_count"].rank(pct=True).fillna(0)) * visibility
baseline_scores = (0.40 * visibility + 0.30 * freshness + 0.25 * position + 0.05 * depth_gap).clip(0, 1).to_numpy()
print(f"Prepared {len(frame):,} eligible rows; declining base rate = {y.mean():.3f}; features = {X.shape[1]}")

Prepared 30,000 eligible rows; declining base rate = 0.542; features = 52


## 2. Split design

I use a client holdout: 20% of pseudonymous clients are assigned to test with seed 42, and every row for those clients stays in test. This tests whether the ranking transfers to unseen clients and prevents client-specific patterns from leaking across the split. The baseline and every model use exactly this same test set.

In [9]:
all_indices = np.arange(len(frame))
clients = frame["client_id"].fillna("unknown").astype(str)
unique_clients = clients.drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
test_clients = set(rng.permutation(unique_clients)[:max(1, int(round(len(unique_clients) * 0.20)))])
test_mask = clients.isin(test_clients).to_numpy()
train_indices = all_indices[~test_mask]
test_indices = all_indices[test_mask]
if y.iloc[train_indices].nunique() < 2 or y.iloc[test_indices].nunique() < 2:
    train_indices, test_indices = train_test_split(all_indices, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
    split_strategy = "stratified_row_holdout"
else:
    split_strategy = "client_holdout"

X_train, X_test = X.iloc[train_indices], X.iloc[test_indices]
y_train, y_test = y.iloc[train_indices], y.iloc[test_indices]
print(f"{split_strategy}: train={len(train_indices):,}, test={len(test_indices):,}, clients in test={len(test_clients):,}")
print(f"train base rate={y_train.mean():.3f}; test base rate={y_test.mean():.3f}")

client_holdout: train=27,675, test=2,325, clients in test=6
train base rate=0.555; test base rate=0.391


## 3. Train + compare vs my baseline

The comparison below uses the test holdout only. `Precision@50` is the primary metric because the practical action is reviewing the highest-risk 50 items; average precision and ROC AUC describe ranking quality across the full holdout. The baseline is the deterministic Week-4 score recomputed above, not a full-data score.

In [10]:
models = {
    "logistic_regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "decision_tree": DecisionTreeClassifier(class_weight="balanced", max_depth=5, min_samples_leaf=50, random_state=RANDOM_STATE),
    "random_forest": RandomForestClassifier(class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25, n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE),
}

def metric_row(name, scores):
    predictions = (scores >= 0.5).astype(int)
    return {
        "model": name,
        "ROC AUC": roc_auc_score(y_test, scores),
        "Average precision": average_precision_score(y_test, scores),
        "Precision@20": precision_at_k(y_test, scores, 20),
        "Precision@50": precision_at_k(y_test, scores, 50),
        "Precision@100": precision_at_k(y_test, scores, 100),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1": f1_score(y_test, predictions, zero_division=0),
    }

scores_by_model = {"baseline_rules": baseline_scores[test_indices]}
results = [metric_row("baseline_rules", scores_by_model["baseline_rules"])]
for name, model in models.items():
    model.fit(X_train, y_train)
    scores_by_model[name] = model.predict_proba(X_test)[:, 1]
    results.append(metric_row(name, scores_by_model[name]))

comparison = pd.DataFrame(results).sort_values(["Precision@50", "Average precision"], ascending=False).reset_index(drop=True)
display(comparison.round(3))
best_model_name = comparison.loc[comparison["model"] != "baseline_rules", "model"].iloc[0]
best_model = models[best_model_name]
print(f"Selected model: {best_model_name} by holdout Precision@50")

,model,ROC AUC,Average precision,Precision@20,Precision@50,Precision@100,Recall,F1
0,random_forest,0.747,0.610,0.70,0.68,0.70,0.741,0.638
1,decision_tree,0.742,0.575,0.55,0.62,0.60,0.716,0.634
2,logistic_regression,0.700,0.522,0.35,0.40,0.44,0.567,0.566
3,baseline_rules,0.628,0.468,0.15,0.22,0.36,0.194,0.279


Selected model: random_forest by holdout Precision@50


## 4. Errors and interpretation

I inspect false positives and false negatives at the model's 0.5 threshold, then use permutation importance on the untouched test holdout. Three examples are shown by pseudonymous ID only. The error slices are descriptive, not proof that any feature causes decline.

In [11]:
best_scores = scores_by_model[best_model_name]
error_frame = frame.iloc[test_indices][["content_id", "client_id", "is_declining_label", "impressions_90d", "avg_position", "content_age_days", "days_since_last_update"]].copy()
error_frame["probability"] = best_scores
error_frame["prediction"] = (best_scores >= 0.5).astype(int)
error_frame["error_type"] = np.select(
    [(error_frame["prediction"] == 1) & (error_frame["is_declining_label"] == 0),
     (error_frame["prediction"] == 0) & (error_frame["is_declining_label"] == 1)],
    ["false_positive", "false_negative"], default="correct"
)
print("Error counts:")
display(error_frame["error_type"].value_counts().rename_axis("error_type").to_frame("rows"))
print("Three highest-confidence mistakes:")
display(error_frame[error_frame["error_type"] != "correct"].sort_values("probability", ascending=False).head(3).round(3))

importance = permutation_importance(best_model, X_test, y_test, scoring="average_precision", n_repeats=3, random_state=RANDOM_STATE, n_jobs=-1)
importance_frame = pd.DataFrame({"feature": X_test.columns, "importance_mean": importance.importances_mean, "importance_std": importance.importances_std}).sort_values("importance_mean", ascending=False).head(10)
print("Top permutation-importance features on the test holdout:")
display(importance_frame.round(4))

if best_model_name == "decision_tree":
    print("Readable tree excerpt:")
    print(export_text(best_model, feature_names=list(X.columns), max_depth=3)[:1800])

print("Interpretation: visibility and freshness features are plausible review signals, but the mistakes show that a declining label is not perfectly separable from aggregate 90-day performance. The model is decision support for prioritization, not an automatic publishing decision.")

Error counts:


,rows
error_type,
correct,1561
false_positive,529
false_negative,235


Three highest-confidence mistakes:


,content_id,client_id,is_declining_label,impressions_90d,avg_position,content_age_days,days_since_last_update,probability,prediction,error_type
23250,content_d2dffcc697a4,client_f74efabef1,0,5091,14.1,144,20,0.737,1,false_positive
23559,content_00603b0349b4,client_f74efabef1,0,1076,25.6,125,20,0.735,1,false_positive
23750,content_e55b8ab078b0,client_f74efabef1,0,369,21.8,112,20,0.734,1,false_positive


Top permutation-importance features on the test holdout:


,feature,importance_mean,importance_std
9,days_with_impressions,0.0825,0.0080
5,log_impressions_90d,0.0302,0.0040
13,ctr,0.0210,0.0056
0,search_volume,0.0101,0.0011
16,scroll_rate,0.0090,0.0059
14,avg_position,0.0079,0.0067
6,log_clicks_90d,0.0074,0.0017
32,age_tier_365+,0.0031,0.0019
10,days_with_sessions,0.0028,0.0002
29,main_intent_unknown,0.0025,0.0002


Interpretation: visibility and freshness features are plausible review signals, but the mistakes show that a declining label is not perfectly separable from aggregate 90-day performance. The model is decision support for prioritization, not an automatic publishing decision.


## Self-check

- [x] Every section is filled with markdown reasoning and executable code.
- [* ] The notebook must be run top to bottom with no errors before submission.
- [x] No client names, URLs, or private queries are used; IDs appear only as pseudonyms in error examples.
- [x] Claims use measured, directional, and decision-support language.
- [* ] Commit `work/notebooks/w05_model.ipynb` to the repo before submitting the repo URL.